# Módulo 04: Limpeza e Preparação de Dados
>
> [Universidade Federal do Ceará (UFC)](https://www.ufc.br/)\
> [Departamento de Computação (DC)](https://dc.ufc.br/pt/)\
> [Capacitação Técnica e Empreendedora em IA (CTE-IA)](https://www.cteia.dc.ufc.br/)\
> Fase I: Capacitação Teórica de IA\
> Disciplina: Programação para Ciência de Dados (60h)\
> Professor: [Lincoln S. Rocha](http://lattes.cnpq.br/0656977742590515)\
> E-mail: <lincoln@dc.ufc.br>
>

Durante a análise e modelagem de dados, uma quantidade significativa de tempo é gasta na preparação dos dados: carregamento, limpeza, transformação e reorganização. Essas tarefas costumam ocupar `80%` ou mais do tempo de um profissional da área de dados. Às vezes, a maneira como os dados são armazenados em arquivos ou bancos de dados não está no formato correto para uma tarefa específica. O Pandas, juntamente com os recursos integrados da linguagem Python, fornece um conjunto de ferramentas de alto nível, flexível e rápido que permite manipular os dados no formato correto. Neste módulo, apresentamos ferramentas para tratar dados ausentes, dados duplicados, manipulação de strings e algumas outras transformações analíticas de dados. Este módulo cobrirá o seguinte conteúdo:

1. Manipulação de Dados Ausentes
2. Transformação de Dados
3. Tipos de Dados de Extensão
4. Manipulação de String
5. Dados Categóricos

## 1. Manipulação de Dados Ausentes

Dados ausentes ocorrem com frequência em muitas aplicações de análise de dados. Um dos objetivos do Pandas é tornar o trabalho com dados ausentes o mais simples possível. Por exemplo, todas as estatísticas descritivas em objetos Pandas excluem dados ausentes por padrão.

### 1.1. Visão Geral

A forma como os dados ausentes são representados em objetos Pandas é um tanto imperfeita, mas é suficiente para a maioria dos usos no mundo real. Para dados com tipo de dados `float64`, o Pandas usa o valor de ponto flutuante `NaN` (*Not a Number*) para representar dados ausentes.

Chamamos isso de *valor sentinela*: quando presente, indica um valor ausente (ou *nulo*):

In [1]:
import numpy as np
import pandas as pd

float_data = pd.Series([1.2, -3.5, np.nan, 0])
float_data

0    1.2
1   -3.5
2    NaN
3    0.0
dtype: float64

O método `isna` nos fornece uma série booleana com `True` onde os valores são nulos:

In [2]:
float_data.isna()

0    False
1    False
2     True
3    False
dtype: bool

No Pandas, adotamos uma convenção usada na linguagem de programação R, referindo-nos aos dados ausentes como `NA`, que significa `"not available"`. Em aplicações estatísticas, dados `NA` podem ser dados inexistentes ou existentes, mas não observados (por exemplo, devido a problemas com a coleta de dados). Ao limpar dados para análise, geralmente é importante analisar os próprios dados ausentes para identificar problemas na coleta de dados ou potenciais vieses nos dados causados ​​por dados ausentes.

O valor interno None do Python também é tratado como `NA`:

In [3]:
string_data = pd.Series(['aardvark', np.nan, None, 'avocado'])
string_data

0    aardvark
1         NaN
2         NaN
3     avocado
dtype: str

In [4]:
string_data.isna()

0    False
1     True
2     True
3    False
dtype: bool

In [5]:
float_data = pd.Series([1, 2, None], dtype='float64')
float_data

0    1.0
1    2.0
2    NaN
dtype: float64

In [6]:
float_data.isna()

0    False
1    False
2     True
dtype: bool

O projeto Pandas tentou tornar o trabalho com dados ausentes consistente em todos os tipos de dados. Funções como `pandas.isna` abstraem muitos dos detalhes incômodos. Veja a tabela abaixo para uma lista de algumas funções relacionadas ao tratamento de dados ausentes.

| Método | Descrição |
|---------|------------|
| `dropna` | Filtra rótulos do eixo com base em valores ausentes. Remove rótulos cujos valores possuem dados faltantes, podendo ajustar o limite de tolerância de dados ausentes. |
| `fillna` | Preenche valores ausentes com um valor especificado ou usando um método de interpolação, como `"ffill"` (preenchimento para frente) ou `"bfill"` (preenchimento para trás). |
| `isna` | Retorna valores booleanos indicando quais elementos são ausentes (`NA`/`NaN`). |
| `notna` | Negação de `isna`; retorna `True` para valores não ausentes e `False` para valores ausentes (`NA`). |

### 1.2. Filtrando Dados Ausentes

Existem algumas maneiras de filtrar dados ausentes. Embora você sempre tenha a opção de fazer isso manualmente usando `pandas.isna` e indexação booleana, `dropna` pode ser útil. Em uma série, ele retorna a série apenas com os dados não nulos e os valores de índice:

In [7]:
data = pd.Series([1, np.nan, 3.5, np.nan, 7])
data

0    1.0
1    NaN
2    3.5
3    NaN
4    7.0
dtype: float64

In [8]:
data.dropna()

0    1.0
2    3.5
4    7.0
dtype: float64

Isso é a mesma coisa que fazer:

In [9]:
data[data.notna()]

0    1.0
2    3.5
4    7.0
dtype: float64

Com objetos `DataFrame`, existem diferentes maneiras de remover dados ausentes. Você pode querer remover linhas ou colunas que sejam todas `NA`, ou apenas aquelas que contenham `NA`. Por padrão, `dropna` remove qualquer linha que contenha um valor ausente:

In [10]:
data = pd.DataFrame([[1., 6.5, 3.], 
                     [1., np.nan, np.nan],
                     [np.nan, np.nan, np.nan], 
                     [np.nan, 6.5, 3.]])
data

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
2,NaN,NaN,NaN
3,NaN,6.5,3.0


In [11]:
data.dropna()

,0,1,2
0,1.0,6.5,3.0


Passando `how="all"` descartará apenas as linhas que são todas `NA`:

In [12]:
data.dropna(how='all')

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
3,NaN,6.5,3.0


**OBS**. Lembre-se de que essas funções retornam novos objetos por padrão e não modificam o conteúdo do objeto original.

Para remover colunas da mesma forma, passe `axis="columns"`:

In [13]:
data[4] = np.nan
data

,0,1,2,4
0,1.0,6.5,3.0,NaN
1,1.0,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,NaN,6.5,3.0,NaN


In [14]:
data.dropna(axis='columns', how='all')

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
2,NaN,NaN,NaN
3,NaN,6.5,3.0


Suponha que você queira manter apenas linhas contendo, no máximo, um certo número de observações ausentes. Você pode indicar isso com o argumento `thresh`:

In [16]:
frame = pd.DataFrame(np.random.standard_normal((7, 3)))

frame.iloc[:4, 1] = np.nan
frame.iloc[:2, 2] = np.nan
frame

,0,1,2
0,0.720185,NaN,NaN
1,1.531754,NaN,NaN
2,0.453046,NaN,-1.330795
3,-1.230084,NaN,0.469179
4,0.999523,0.723495,-1.948597
5,-0.370828,-0.090245,-0.118006
6,0.785932,0.070947,0.237913


In [17]:
frame.dropna()

,0,1,2
4,0.999523,0.723495,-1.948597
5,-0.370828,-0.090245,-0.118006
6,0.785932,0.070947,0.237913


In [18]:
frame.dropna(thresh=2)

,0,1,2
2,0.453046,NaN,-1.330795
3,-1.230084,NaN,0.469179
4,0.999523,0.723495,-1.948597
5,-0.370828,-0.090245,-0.118006
6,0.785932,0.070947,0.237913


### 1.3. Preenchendo Dados Faltantes

Em vez de filtrar os dados ausentes (e potencialmente descartar outros dados junto com eles), você pode querer preencher as "lacunas" de diversas maneiras. Para a maioria dos propósitos, o método `fillna` é a função mais utilizada. Chamar `fillna` com uma constante substitui os valores ausentes por esse valor:

In [19]:
frame.fillna(0)

,0,1,2
0,0.720185,0.000000,0.000000
1,1.531754,0.000000,0.000000
2,0.453046,0.000000,-1.330795
3,-1.230084,0.000000,0.469179
4,0.999523,0.723495,-1.948597
5,-0.370828,-0.090245,-0.118006
6,0.785932,0.070947,0.237913


Ao chamar `fillna` com um dicionário, você pode usar um valor de preenchimento diferente para cada coluna:

In [20]:
frame.fillna({1: 0.5, 2: 0})

,0,1,2
0,0.720185,0.500000,0.000000
1,1.531754,0.500000,0.000000
2,0.453046,0.500000,-1.330795
3,-1.230084,0.500000,0.469179
4,0.999523,0.723495,-1.948597
5,-0.370828,-0.090245,-0.118006
6,0.785932,0.070947,0.237913


Os mesmos métodos de interpolação disponíveis para reindexação podem ser usados para fazer o preenchimento de dados ausentes:

In [21]:
frame = pd.DataFrame(np.random.standard_normal((6, 3)))

frame.iloc[2:, 1] = np.nan
frame.iloc[4:, 2] = np.nan
frame

,0,1,2
0,0.260791,-0.157065,-0.526992
1,-0.924267,0.090824,-0.554392
2,0.168913,NaN,0.182440
3,0.206433,NaN,0.071627
4,-0.597715,NaN,NaN
5,1.596322,NaN,NaN


In [22]:
frame.ffill()

,0,1,2
0,0.260791,-0.157065,-0.526992
1,-0.924267,0.090824,-0.554392
2,0.168913,0.090824,0.182440
3,0.206433,0.090824,0.071627
4,-0.597715,0.090824,0.071627
5,1.596322,0.090824,0.071627


In [23]:
frame.loc[5, 2] = 0.222333
frame.loc[5, 1] = 1.4168944
frame

,0,1,2
0,0.260791,-0.157065,-0.526992
1,-0.924267,0.090824,-0.554392
2,0.168913,NaN,0.182440
3,0.206433,NaN,0.071627
4,-0.597715,NaN,NaN
5,1.596322,1.416894,0.222333


In [24]:
frame.bfill()

,0,1,2
0,0.260791,-0.157065,-0.526992
1,-0.924267,0.090824,-0.554392
2,0.168913,1.416894,0.182440
3,0.206433,1.416894,0.071627
4,-0.597715,1.416894,0.222333
5,1.596322,1.416894,0.222333


Se `limit` for especificado, ele definirá o número máximo de valores `NaN` consecutivos a serem preenchidos para frente/para trás. Em outras palavras, se houver uma lacuna com mais do que esse número de `NaN`s consecutivos, ela será preenchida apenas parcialmente. Se o método não for especificado, este é o número máximo de entradas ao longo de todo o eixo onde os `NaN`s serão preenchidos. Deve ser maior que `0`, caso contrário, `None`. Por exemplo:

In [25]:
frame.ffill(limit=2)

,0,1,2
0,0.260791,-0.157065,-0.526992
1,-0.924267,0.090824,-0.554392
2,0.168913,0.090824,0.182440
3,0.206433,0.090824,0.071627
4,-0.597715,NaN,0.071627
5,1.596322,1.416894,0.222333


In [26]:
frame.bfill(limit=2)

,0,1,2
0,0.260791,-0.157065,-0.526992
1,-0.924267,0.090824,-0.554392
2,0.168913,NaN,0.182440
3,0.206433,1.416894,0.071627
4,-0.597715,1.416894,0.222333
5,1.596322,1.416894,0.222333


Com o `fillna` você pode fazer muitas outras coisas, como imputação simples de dados usando estatísticas de mediana ou média:

In [27]:
data = pd.Series([1., np.nan, 3.5, np.nan, 7]) 
data

0    1.0
1    NaN
2    3.5
3    NaN
4    7.0
dtype: float64

In [28]:
data.fillna(data.mean())

0    1.000000
1    3.833333
2    3.500000
3    3.833333
4    7.000000
dtype: float64

A tabela abaixo apresenta uma referência sobre os argumentos da função `fillna`.

| Argumento | Descrição |
|------------|------------|
| `value` | Valor escalar ou objeto semelhante a dicionário usado para preencher valores ausentes. |
| `method` | Método de interpolação: pode ser `"bfill"` (preenchimento para trás) ou `"ffill"` (preenchimento para frente); o padrão é `None`. |
| `axis` | Eixo no qual o preenchimento será aplicado (`"index"` ou `"columns"`); o padrão é `axis="index"`. |
| `limit` | Para preenchimentos para frente e para trás, define o número máximo de períodos consecutivos a serem preenchidos. |

## 2. Transformação de Dados

Até agora, nos concentramos em lidar com dados ausentes. Filtragem, limpeza e outras transformações são outra classe de operações importantes.

### 2.1. Removendo Duplicatas

Linhas duplicadas podem ser encontradas em um `DataFrame` por vários motivos. Aqui está um exemplo:

In [30]:
data = pd.DataFrame({'k1': ['one', 'two'] * 3 + ['two'],
                    'k2': [1, 1, 2, 3, 3, 4, 4]})
data

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4
6,two,4


O método `duplicated` do `DataFrame` retorna uma série booleana indicando se cada linha é uma duplicata (seus valores de coluna são exatamente iguais aos de uma linha anterior) ou não:

In [31]:
data.duplicated()

0    False
1    False
2    False
3    False
4    False
5    False
6     True
dtype: bool

Da mesma forma, `drop_duplicates` retorna um `DataFrame` com linhas onde o array duplicado é filtrado como `False`:

In [32]:
data.drop_duplicates()

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4


Ambos os métodos consideram, por padrão, todas as colunas; alternativamente, você pode especificar qualquer subconjunto delas para detectar duplicatas. Suponha que tenhamos uma coluna adicional de valores e queiramos filtrar duplicatas com base apenas na coluna `"k1"`:

In [33]:
data['v1'] = range(7)
data

,k1,k2,v1
0,one,1,0
1,two,1,1
2,one,2,2
3,two,3,3
4,one,3,4
5,two,4,5
6,two,4,6


In [34]:
data.drop_duplicates(subset=['k1'])

,k1,k2,v1
0,one,1,0
1,two,1,1


`duplicated` e `drop_duplicates` mantêm, por padrão, a primeira combinação de valores observada. Passar `keep="last"` retornará a última:

In [35]:
data.drop_duplicates(subset=['k1'], keep='last') # compara os valores somente nas colunas em "subset"

,k1,k2,v1
4,one,3,4
6,two,4,6


In [36]:
data.drop_duplicates(['k1', 'k2'], keep='last')

,k1,k2,v1
0,one,1,0
1,two,1,1
2,one,2,2
3,two,3,3
4,one,3,4
6,two,4,6


### 2.2. Transformando Dados Usando uma Função ou Mapeamento

Para muitos conjuntos de dados, você pode desejar realizar alguma transformação com base nos valores de um array, série ou coluna em um `DataFrame`. Considere os seguintes dados hipotéticos coletados sobre vários tipos de carne:

In [40]:
data = pd.DataFrame({'food': ['bacon', 'pulled pork', 'bacon',
                              'pastrami', 'corned beef', 'bacon',
                              'pastrami', 'honey ham', 'nova lox'],
                    'ounces': [4, 3, 12, 6, 7.5, 8, 3, 5, 6]})
data

,food,ounces
0,bacon,4.0
1,pulled pork,3.0
2,bacon,12.0
3,pastrami,6.0
4,corned beef,7.5
5,bacon,8.0
6,pastrami,3.0
7,honey ham,5.0
8,nova lox,6.0


Suponha que você queira adicionar uma coluna indicando o tipo de animal de onde cada alimento veio. Vamos mapear cada tipo distinto de carne para o tipo de animal:

In [41]:
meat_to_animal = {
    'bacon': 'pig',
    'pulled pork': 'pig',
    'pastrami': 'cow',
    'corned beef': 'cow',
    'honey ham': 'pig',
    'nova lox': 'salmon'
}

O método map em uma `Series` aceita uma função ou objeto semelhante a um dicionário contendo um mapeamento para fazer a transformação de valores:

In [42]:
data['animal'] = data['food'].map(meat_to_animal)
data

,food,ounces,animal
0,bacon,4.0,pig
1,pulled pork,3.0,pig
2,bacon,12.0,pig
3,pastrami,6.0,cow
4,corned beef,7.5,cow
5,bacon,8.0,pig
6,pastrami,3.0,cow
7,honey ham,5.0,pig
8,nova lox,6.0,salmon


Também poderíamos ter passado uma função (ou expressão lambda) que faz todo o trabalho:

In [43]:
def get_animal(x):
    return meat_to_animal[x]

data['food'].map(get_animal)
data

,food,ounces,animal
0,bacon,4.0,pig
1,pulled pork,3.0,pig
2,bacon,12.0,pig
3,pastrami,6.0,cow
4,corned beef,7.5,cow
5,bacon,8.0,pig
6,pastrami,3.0,cow
7,honey ham,5.0,pig
8,nova lox,6.0,salmon


In [44]:
data['food'].map(lambda x: meat_to_animal[x])

0       pig
1       pig
2       pig
3       cow
4       cow
5       pig
6       cow
7       pig
8    salmon
Name: food, dtype: str

Usar o `map` é uma maneira conveniente de realizar transformações por elementos e outras operações relacionadas à limpeza de dados.

### 2.3. Substituindo Valores

Preencher dados ausentes com o método `fillna` é um caso especial de substituição de valores mais geral. Como você já viu, `map` pode ser usado para modificar um subconjunto de valores em um objeto, mas `replace` fornece uma maneira mais simples e flexível de fazer isso. Vamos considerar esta série:

In [ ]:
data = pd.Series([1., -999., 2., -999., -1000., 3.])
data

0       1.0
1    -999.0
2       2.0
3    -999.0
4   -1000.0
5       3.0
dtype: float64

Os valores `-999` podem ser valores sentinela para dados ausentes. Para substituí-los por valores `NA` que o Pandas entende, podemos usar `replace`, produzindo uma nova série:

In [46]:
data.replace(-999, np.nan)

0       1.0
1       NaN
2       2.0
3       NaN
4   -1000.0
5       3.0
dtype: float64

Se você quiser substituir vários valores de uma só vez, passe uma lista e depois o valor substituto:

In [47]:
data.replace([-999, -1000], np.nan)

0    1.0
1    NaN
2    2.0
3    NaN
4    NaN
5    3.0
dtype: float64

Para usar uma substituição diferente para cada valor, passe uma lista de substitutos:

In [48]:
data.replace([-999, -1000], [np.nan, 0])

0    1.0
1    NaN
2    2.0
3    NaN
4    0.0
5    3.0
dtype: float64

O argumento passado também pode ser um dicionário:

In [49]:
data.replace({-999: np.nan, -1000: 0})

0    1.0
1    NaN
2    2.0
3    NaN
4    0.0
5    3.0
dtype: float64

### 2.4. Renomeando Índices de Eixo

Assim como os valores em uma série, os rótulos dos eixos podem ser transformados de forma semelhante por uma função ou mapeamento de alguma forma para produzir novos objetos com rótulos diferentes. Você também pode modificar os eixos no local sem criar uma nova estrutura de dados. Veja um exemplo simples:

In [51]:
data = pd.DataFrame(np.arange(12).reshape((3, 4)),
                    index=['Ohio', 'Colorado', 'New York'],
                    columns=['one', 'two', 'three', 'four'])

data

,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7
New York,8,9,10,11


Assim como uma série, os índices do eixo têm um método `map`:

In [52]:
def transform(x):
    return x[:4].upper()

data.index.map(transform)

Index(['OHIO', 'COLO', 'NEW '], dtype='str')

Você pode atribuir ao atributo `index`, modificando o `DataFrame` corrente:

In [53]:
data.index = data.index.map(transform)

data

,one,two,three,four
OHIO,0,1,2,3
COLO,4,5,6,7
NEW,8,9,10,11


Se você quiser criar uma versão transformada de um conjunto de dados sem modificar o original, um método útil é `rename`:

In [54]:
data.rename(index=str.title, columns=str.upper)

,ONE,TWO,THREE,FOUR
Ohio,0,1,2,3
Colo,4,5,6,7
New,8,9,10,11


Notavelmente, o método `rename` pode ser usado em conjunto com um objeto semelhante a um dicionário, fornecendo novos valores para um subconjunto dos rótulos dos eixos:

In [55]:
data.rename(index={'OHIO': 'INDIANA'},
            columns={'three': 'peekaboo'})

,one,two,peekaboo,four
INDIANA,0,1,2,3
COLO,4,5,6,7
NEW,8,9,10,11


`rename` evita que você tenha que copiar o `DataFrame` manualmente e atribuir
novos valores aos seus atributos de índice e colunas.

### 2.5. Discretização e Binning

Dados contínuos são frequentemente discretizados ou separados em "grupos" ("bins") para análise. Suponha que você tenha dados sobre um grupo de pessoas em um estudo e queira agrupá-los em grupos de idade discretos:

In [56]:
ages = [20, 22, 25, 27, 21, 23, 37, 31, 61, 45, 41, 32]

Vamos dividi-los em grupos de 18 a 25, 26 a 35, 36 a 60 e, finalmente, 61 anos ou mais. Para isso, você precisa usar `pandas.cut`:

In [57]:
bins = [18, 25, 35, 60, 100]

age_categories = pd.cut(ages, bins)
age_categories

[(18, 25], (18, 25], (18, 25], (25, 35], (18, 25], ..., (25, 35], (60, 100], (35, 60], (35, 60], (25, 35]]
Length: 12
Categories (4, interval[int64, right]): [(18, 25] < (25, 35] < (35, 60] < (60, 100]]

O objeto retornado pelo pandas é um objeto `Categorical` especial. A saída que você vê descreve os grupos (bins) computados por `pandas.cut`. Cada grupo é identificado por um tipo de valor de intervalo especial (exclusivo do pandas) contendo os limites inferior e superior de cada grupo:

In [58]:
age_categories.codes

array([0, 0, 0, 1, 0, 0, 2, 1, 3, 2, 2, 1], dtype=int8)

In [59]:
age_categories.categories

IntervalIndex([(18, 25], (25, 35], (35, 60], (60, 100]], dtype='interval[int64, right]')

In [60]:
age_categories.categories[0]

Interval(18, 25, closed='right')

In [61]:
pd.Series(age_categories).value_counts()

(18, 25]     5
(25, 35]     3
(35, 60]     3
(60, 100]    1
Name: count, dtype: int64

Observe que `pd.Series(age_categories).value_counts()` são as contagens de grupos para o resultado de `pandas.cut`.

Na representação em string de um intervalo, um parêntese significa que o lado está aberto (exclusivo), enquanto o colchete significa que está fechado (inclusivo). Você pode alterar qual lado está fechado passando `right=False`:

In [62]:
pd.cut(ages, bins, right=False)

[[18, 25), [18, 25), [25, 35), [25, 35), [18, 25), ..., [25, 35), [60, 100), [35, 60), [35, 60), [25, 35)]
Length: 12
Categories (4, interval[int64, left]): [[18, 25) < [25, 35) < [35, 60) < [60, 100)]

Você pode substituir a rotulagem de grupo (bin) baseada em intervalo padrão passando uma lista ou array para a opção labels:

In [63]:
group_names = ['Youth', 'YoungAdult', 'MiddleAged', 'Senior']

pd.cut(ages, bins, labels=group_names)

['Youth', 'Youth', 'Youth', 'YoungAdult', 'Youth', ..., 'YoungAdult', 'Senior', 'MiddleAged', 'MiddleAged', 'YoungAdult']
Length: 12
Categories (4, str): ['Youth' < 'YoungAdult' < 'MiddleAged' < 'Senior']

Se você passar um número inteiro de grupos (bins) para `pandas.cut` em vez de bordas de compartimento explícitas, ele calculará grupos (bins) de comprimento igual com base nos valores mínimo e máximo nos dados. Considere o caso de alguns dados uniformemente distribuídos divididos em quartos:

In [64]:
data = np.random.uniform(size=20)

pd.cut(data, 4, precision=2)

[(0.5, 0.73], (0.022, 0.26], (0.5, 0.73], (0.5, 0.73], (0.26, 0.5], ..., (0.5, 0.73], (0.022, 0.26], (0.022, 0.26], (0.73, 0.97], (0.022, 0.26]]
Length: 20
Categories (4, interval[float64, right]): [(0.022, 0.26] < (0.26, 0.5] < (0.5, 0.73] < (0.73, 0.97]]

A opção `precision=2` limita a precisão decimal a dois dígitos.

Uma função intimamente relacionada, `pandas.qcut`, agrupa os dados com base em quartis de amostra. Dependendo da distribuição dos dados, o uso de `pandas.cut` geralmente não resultará em cada caixa com o mesmo número de pontos de dados. Como `pandas.qcut` usa quartis de amostra, você obterá caixas com tamanhos aproximadamente iguais:

In [65]:
data = np.random.standard_normal(1000)

quartiles = pd.qcut(data, 4, precision=2)
quartiles

[(-0.023, 0.63], (-0.73, -0.023], (-0.023, 0.63], (-0.023, 0.63], (-2.82, -0.73], ..., (-2.82, -0.73], (-0.73, -0.023], (0.63, 2.86], (-0.73, -0.023], (-0.73, -0.023]]
Length: 1000
Categories (4, interval[float64, right]): [(-2.82, -0.73] < (-0.73, -0.023] < (-0.023, 0.63] < (0.63, 2.86]]

In [66]:
pd.Series(quartiles).value_counts()

(-2.82, -0.73]     250
(-0.73, -0.023]    250
(-0.023, 0.63]     250
(0.63, 2.86]       250
Name: count, dtype: int64

Semelhante ao `pandas.cut`, você pode passar seus próprios quartis (números entre 0 e 1, inclusive):

In [67]:
pd.qcut(data, [0, 0.1, 0.5, 0.9, 1]).value_counts()

(-2.807, -1.264]     100
(-1.264, -0.0226]    400
(-0.0226, 1.293]     400
(1.293, 2.864]       100
Name: count, dtype: int64

Voltaremos a falar de `pandas.cut` e `pandas.qcut` mais adiante neste módulo durante nossa discussão sobre operações de agregação e grupo, pois essas funções de discretização são especialmente úteis para análise de quartis e grupos.

### 2.6. Detectando e Filtrando Outliers

Filtrar ou transformar outliers é, em grande parte, uma questão de aplicar operações de array. Considere um `DataFrame` com alguns dados distribuídos normalmente:

In [68]:
data = pd.DataFrame(np.random.standard_normal((1000, 4)))

data.describe()

,0,1,2,3
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,-0.038598,0.008798,-0.014861,0.069522
std,1.016408,1.032892,0.966335,0.989393
min,-3.146322,-2.827821,-3.044995,-3.201159
25%,-0.747515,-0.701673,-0.677009,-0.595031
50%,0.004237,0.019492,0.007806,0.109995
75%,0.652455,0.711779,0.634819,0.738232
max,3.141137,4.050352,3.001740,4.182425


Suponha que você queira encontrar valores em uma das colunas que excedam `3` em valor absoluto:

In [69]:
col = data[2]
col[col.abs() > 3]

98    -3.044995
250    3.001740
Name: 2, dtype: float64

Para selecionar todas as linhas com um valor superior a `3` ou `–3`, você pode usar o método `any` em um `DataFrame` booleano:

In [70]:
data[(data.abs() > 3).any(axis='columns')]

,0,1,2,3
92,-1.623466,1.190565,-0.531440,3.057852
98,1.586357,-0.020029,-3.044995,-0.012607
153,-0.567875,3.365178,-0.485703,-0.198513
217,0.513935,3.492154,0.714952,0.237140
250,0.700493,0.735145,3.001740,1.210783
538,-0.315196,4.050352,0.010496,-2.172033
542,-1.548036,3.478702,0.232744,1.206552
603,3.141137,-1.604800,-1.009760,-0.796591
753,-3.146322,0.459395,-1.244507,2.139609
854,1.276512,-2.250885,-0.587798,4.182425


Os parênteses em torno de `data.abs() > 3` são necessários para chamar o método `any` no resultado da operação de comparação.

Os valores podem ser definidos com base nesses critérios. Aqui está o código para limitar os valores fora do intervalo `–3` a `3`:

In [71]:
data[data.abs() > 3] = np.sign(data) * 3

data.describe()

,0,1,2,3
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,-0.038593,0.006412,-0.014818,0.068386
std,1.015539,1.025115,0.966189,0.984054
min,-3.000000,-2.827821,-3.000000,-3.000000
25%,-0.747515,-0.701673,-0.677009,-0.595031
50%,0.004237,0.019492,0.007806,0.109995
75%,0.652455,0.711779,0.634819,0.738232
max,3.000000,3.000000,3.000000,3.000000


O comando `np.sign(data)` produz valores `1` e `–1` com base nos valores em data serem positivos ou negativos:

In [72]:
np.sign(data).head()

,0,1,2,3
0,-1.0,-1.0,-1.0,1.0
1,-1.0,1.0,1.0,1.0
2,-1.0,-1.0,-1.0,-1.0
3,1.0,1.0,-1.0,1.0
4,-1.0,1.0,1.0,1.0


### 2.7. Permutação e Amostragem Aleatória

É possível permutar (reordenar aleatoriamente) uma série ou as linhas de um `DataFrame` usando a função `numpy.random.permutation`. Chamar a função `permutation` com o comprimento do eixo que você deseja permutar produz um array de inteiros indicando a nova ordenação:

In [73]:
frame = pd.DataFrame(np.arange(5 * 7).reshape((5, 7)))
frame

,0,1,2,3,4,5,6
0,0,1,2,3,4,5,6
1,7,8,9,10,11,12,13
2,14,15,16,17,18,19,20
3,21,22,23,24,25,26,27
4,28,29,30,31,32,33,34


In [74]:
sampler = np.random.permutation(5)
sampler

array([3, 0, 2, 1, 4])

Esse array pode então ser usada na indexação baseada em `iloc` ou na função `take` equivalente:

In [75]:
frame.take(sampler)

,0,1,2,3,4,5,6
3,21,22,23,24,25,26,27
0,0,1,2,3,4,5,6
2,14,15,16,17,18,19,20
1,7,8,9,10,11,12,13
4,28,29,30,31,32,33,34


In [76]:
frame.iloc[sampler]

,0,1,2,3,4,5,6
3,21,22,23,24,25,26,27
0,0,1,2,3,4,5,6
2,14,15,16,17,18,19,20
1,7,8,9,10,11,12,13
4,28,29,30,31,32,33,34


Ao invocar `take` com `axis="columns"`, também poderíamos selecionar uma permutação das colunas:

In [77]:
column_sampler = np.random.permutation(7)
column_sampler

array([5, 1, 2, 0, 6, 4, 3])

In [78]:
frame.take(column_sampler, axis='columns')

,5,1,2,0,6,4,3
0,5,1,2,0,6,4,3
1,12,8,9,7,13,11,10
2,19,15,16,14,20,18,17
3,26,22,23,21,27,25,24
4,33,29,30,28,34,32,31


Para selecionar um subconjunto aleatório sem substituição (a mesma linha não pode aparecer duas vezes), você pode usar o método `sample` em `Series` e `DataFrame`:

In [81]:
frame.sample(n=3)

,0,1,2,3,4,5,6
1,7,8,9,10,11,12,13
3,21,22,23,24,25,26,27
2,14,15,16,17,18,19,20


Para gerar uma amostra com substituição (para permitir escolhas repetidas), passe `replace=True` para `sample`:

In [82]:
choices = pd.Series([5, 7, -1, 6, 4])
choices.sample(n=10, replace=True)

0    5
4    4
0    5
4    4
3    6
3    6
0    5
0    5
1    7
2   -1
dtype: int64

### 2.8. Calculando Indicadores/Variáveis ​​Fictícias

Outro tipo de transformação para modelagem estatística ou aplicações de aprendizado de máquina é converter uma variável categórica em uma matriz fictícia (*dummy*) ou indicadora (*indicator*). Se uma coluna em um `DataFrame` tiver `k` valores distintos, você derivaria uma matriz ou `DataFrame` com `k` colunas contendo todos os valores `1` e `0`. O Pandas possui uma função `pandas.get_dummies` para isso, embora você também possa criar uma. Vamos considerar um exemplo de `DataFrame`:

In [83]:
frame = pd.DataFrame({'key': ['b', 'b', 'a', 'c', 'a', 'b'],
                      'data1': range(6)})

frame

,key,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,b,5


In [84]:
pd.get_dummies(frame['key'], dtype=int)

,a,b,c
0,0,1,0
1,0,1,0
2,1,0,0
3,0,0,1
4,1,0,0
5,0,1,0


Em alguns casos, você pode querer adicionar um prefixo às colunas no `DataFrame` do indicador, que pode então ser mesclado com os outros dados. `pandas.get_dummies` tem um argumento de prefixo (`prefix`) para fazer isso:

In [ ]:
dummies = pd.get_dummies(frame['key'], prefix='key', dtype=int)

frame_with_dummy = frame[['data1']].join(dummies)
frame_with_dummy

,data1,key_a,key_b,key_c
0,0,0,1,0
1,1,0,1,0
2,2,1,0,0
3,3,0,0,1
4,4,1,0,0
5,5,0,1,0


O método `DataFrame.join` será explicado com mais detalhes no próximo módulo.

Se uma linha em um `DataFrame` pertencer a várias categorias, teremos que usar uma abordagem diferente para criar as variáveis ​​fictícias. Vejamos o dataset MovieLens 1M:

In [85]:
movie_names = ['movie_id', 'title', 'genres']

movies = pd.read_table('./datasets/movies.dat', sep='::', header=None, names=movie_names, engine='python')
movies.head(10)

,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children's
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller


O Pandas implementou um método especial da `Series` `str.get_dummies` que lida com este cenário de associação de vários grupos codificados como uma string delimitada:

In [86]:
dummies = movies['genres'].str.get_dummies('|')
dummies.iloc[:10, :6]

,Action,Adventure,Animation,Children's,Comedy,Crime
0,0,0,1,1,1,0
1,0,1,0,1,0,0
2,0,0,0,0,1,0
3,0,0,0,0,1,0
4,0,0,0,0,1,0
5,1,0,0,0,0,1
6,0,0,0,0,1,0
7,0,1,0,1,0,0
8,1,0,0,0,0,0
9,1,1,0,0,0,0


Então, como antes, você pode combinar isso com filmes enquanto adiciona um `"Genre_"` aos nomes das colunas no `DataFrame` dos `dummies` com o método `add_prefix`:

In [87]:
movies_with_idx = movies.join(dummies.add_prefix('Genre_'))
movies_with_idx.iloc[0]

movie_id                                       1
title                           Toy Story (1995)
genres               Animation|Children's|Comedy
Genre_Action                                   0
Genre_Adventure                                0
Genre_Animation                                1
Genre_Children's                               1
Genre_Comedy                                   1
Genre_Crime                                    0
Genre_Documentary                              0
Genre_Drama                                    0
Genre_Fantasy                                  0
Genre_Film-Noir                                0
Genre_Horror                                   0
Genre_Musical                                  0
Genre_Mystery                                  0
Genre_Romance                                  0
Genre_Sci-Fi                                   0
Genre_Thriller                                 0
Genre_War                                      0
Genre_Western       

Uma receita útil para aplicações estatísticas é combinar `pandas.get_dummies` com uma função de discretização como `pandas.cut`:

In [88]:
np.random.seed(42)
values = np.random.uniform(size=10)

values

array([0.37454012, 0.95071431, 0.73199394, 0.59865848, 0.15601864,
       0.15599452, 0.05808361, 0.86617615, 0.60111501, 0.70807258])

In [89]:
bins = [0, 0.2, 0.4, 0.6, 0.8, 1]

pd.get_dummies(pd.cut(values, bins), dtype=int)

,"(0.0, 0.2]","(0.2, 0.4]","(0.4, 0.6]","(0.6, 0.8]","(0.8, 1.0]"
0,0,1,0,0,0
1,0,0,0,0,1
2,0,0,0,1,0
3,0,0,1,0,0
4,1,0,0,0,0
5,1,0,0,0,0
6,1,0,0,0,0
7,0,0,0,0,1
8,0,0,0,1,0
9,0,0,0,1,0


## 3. Tipos de Dados de Extensão

O Pandas foi originalmente desenvolvido com base nos recursos presentes no NumPy, uma biblioteca de computação de arrays usada principalmente para trabalhar com dados numéricos. Muitos conceitos do Pandas, como dados ausentes, foram implementados usando o que estava disponível no NumPy, enquanto se tentava maximizar a compatibilidade entre as bibliotecas que usavam o NumPy e o Pandas juntos.

A construção com base no NumPy levou a uma série de deficiências, como:

- O tratamento de dados ausentes para alguns tipos de dados numéricos, como inteiros e booleanos, era incompleto. Como resultado, quando dados ausentes eram introduzidos nesses dados, o Pandas convertia o tipo de dado para `float64` e usava `np.nan` para representar
valores nulos. Isso teve efeitos cumulativos, introduzindo problemas sutis em muitos
algoritmos do Pandas.

- Conjuntos de dados com muitos dados de string eram computacionalmente caros e consumiam muita memória.

- Alguns tipos de dados, como intervalos de tempo, deltas de tempo e carimbos de data/hora com fusos horários, não podiam ser suportados de forma eficiente sem o uso de matrizes de objetos Python computacionalmente caras.

Mais recentemente, o Pandas desenvolveu um sistema de tipos de extensão que permite a adição de novos tipos de dados, mesmo que não sejam suportados nativamente pelo NumPy. Esses novos tipos de dados podem ser tratados como de primeira classe, juntamente com os dados provenientes de matrizes NumPy.

Vejamos um exemplo em que criamos uma série de inteiros com um valor ausente:

In [90]:
s = pd.Series([1, 2, 3, None])
s

0    1.0
1    2.0
2    3.0
3    NaN
dtype: float64

In [91]:
s.dtype

dtype('float64')

Principalmente por questões de compatibilidade com versões anteriores, `Series` usa o comportamento legado de usar um tipo de dado `float64` e `np.nan` para o valor ausente. Poderíamos criar esta `Series` usando `pandas.Int64Dtype`:

In [92]:
s = pd.Series([1, 2, 3, None], dtype=pd.Int64Dtype())
s

0       1
1       2
2       3
3    <NA>
dtype: Int64

In [93]:
s.isna()

0    False
1    False
2    False
3     True
dtype: bool

A saída `<NA>` indica que um valor está ausente para uma matriz de tipo de extensão. Isso
usa o valor sentinela especial `pandas.NA`:

In [94]:
s[3]

<NA>

In [95]:
s[3] is pd.NA

True

Também poderíamos ter usado a abreviação `"Int64"` em vez de `pd.Int64Dtype()` para especificar o tipo. A capitalização é necessária, caso contrário, será um tipo não-extensível baseado em NumPy:

In [96]:
s = pd.Series([1, 2, 3, None], dtype='Int64')
s

0       1
1       2
2       3
3    <NA>
dtype: Int64

O pandas também tem um tipo de extensão especializado para dados de string que não usa matrizes de objetos NumPy (ele requer a biblioteca `pyarrow`, que você pode precisar instalar separadamente):

In [97]:
s = pd.Series(['one', 'two', None, 'three'], dtype=pd.StringDtype())
s

0      one
1      two
2     <NA>
3    three
dtype: string

Essas matrizes de strings geralmente usam muito menos memória e são frequentemente mais eficientes computacionalmente para realizar operações em grandes conjuntos de dados.

Os tipos de extensão podem ser passados ​​para o método `astype` da série, permitindo a conversão facilmente como parte do seu processo de limpeza de dados:

In [98]:
frame = pd.DataFrame({'A': [1, 2, None, 4],
                      'B': ['one', 'two', 'three', None],
                      'C': [False, None, False, True]})

frame

,A,B,C
0,1.0,one,False
1,2.0,two,None
2,NaN,three,False
3,4.0,NaN,True


In [99]:
frame['A'] = frame['A'].astype('Int64')
frame['B'] = frame['B'].astype('string')
frame['C'] = frame['C'].astype('boolean')

frame

,A,B,C
0,1,one,False
1,2,two,<NA>
2,<NA>,three,False
3,4,<NA>,True


A tabela abaixo apresenta uma série de tipos de dados de extensão do Pandas.

| Tipo de Extensão | Descrição |
|------------------|------------|
| `BooleanDtype` | Dados booleanos anuláveis; use `"boolean"` ao passar como string. |
| `CategoricalDtype` | Tipo de dado categórico; use `"category"` ao passar como string. |
| `DatetimeTZDtype` | Tipo de data e hora com informação de fuso horário. |
| `Float32Dtype` | Ponto flutuante anulável de 32 bits; use `"Float32"` ao passar como string. |
| `Float64Dtype` | Ponto flutuante anulável de 64 bits; use `"Float64"` ao passar como string. |
| `Int8Dtype` | Inteiro assinado anulável de 8 bits; use `"Int8"` ao passar como string. |
| `Int16Dtype` | Inteiro assinado anulável de 16 bits; use `"Int16"` ao passar como string. |
| `Int32Dtype` | Inteiro assinado anulável de 32 bits; use `"Int32"` ao passar como string. |
| `Int64Dtype` | Inteiro assinado anulável de 64 bits; use `"Int64"` ao passar como string. |
| `UInt8Dtype` | Inteiro não assinado anulável de 8 bits; use `"UInt8"` ao passar como string. |
| `UInt16Dtype` | Inteiro não assinado anulável de 16 bits; use `"UInt16"` ao passar como string. |
| `UInt32Dtype` | Inteiro não assinado anulável de 32 bits; use `"UInt32"` ao passar como string. |
| `UInt64Dtype` | Inteiro não assinado anulável de 64 bits; use `"UInt64"` ao passar como string. |

## 4. Manipulação de String

Python é há muito tempo uma linguagem popular para manipulação de dados brutos, em parte devido à sua facilidade de uso para processamento de strings e texto. A maioria das operações de texto é simplificada com os métodos integrados do objeto string. Para correspondências de padrões e manipulações de texto mais complexas, expressões regulares podem ser necessárias. O Pandas contribui para a solução, permitindo que você aplique strings e expressões regulares de forma concisa em matrizes inteiras de dados, além de lidar com o incômodo da falta de dados.

### 4.1. Métodos de String Integrados em Python

Em muitas aplicações de manipulação e script de strings, métodos de string integrados são suficientes. Por exemplo, uma string separada por vírgulas pode ser dividida em partes com `split`:

In [ ]:
val = 'a,b, guido'

`split` é frequentemente combinado com `strip` para cortar espaços em branco (incluindo quebras de linha):

In [2]:
pieces = [x.strip() for x in val.split(',')]
pieces

['a', 'b', 'guido']

Essas substrings podem ser concatenadas com um delimitador de dois pontos usando adição:

In [3]:
first, second, third = pieces
first + '::' + second + '::' + third

'a::b::guido'

Mas este não é um método genérico prático. Uma maneira mais rápida e em Python é passar uma lista ou tupla para o método `join` na string `"::"`:

In [4]:
'::'.join(pieces)

'a::b::guido'

Outros métodos se preocupam em localizar substrings. Usar a palavra-chave `in` do Python é a melhor maneira de detectar uma substring, embora `index` e `find` também possam ser usados:

In [5]:
'guido' in val

True

In [6]:
val.index(',')

1

In [10]:
val.find(':')

-1

Observe que a diferença entre `find` e `index` é que index gera uma exceção se a string não for encontrada (em vez de retornar `–1`):

In [107]:
val.index(":")

ValueError: substring not found

Da mesma forma, count retorna o número de ocorrências de uma substring específica:

In [8]:
val.count(',')

2

`replace` substituirá ocorrências de um padrão por outro. Também é comumente usado para excluir padrões, passando uma string vazia:

In [11]:
val.replace(',', '::')

'a::b:: guido'

In [12]:
val.replace(',', '')

'ab guido'

A tabela abaixo lista de alguns métodos de string do Python.

| Método | Descrição |
|---------|------------|
| `count` | Retorna o número de ocorrências não sobrepostas de uma substring dentro da string. |
| `endswith` | Retorna `True` se a string termina com o sufixo especificado. |
| `startswith` | Retorna `True` se a string começa com o prefixo especificado. |
| `join` | Usa a string como delimitador para concatenar uma sequência de outras strings. |
| `index` | Retorna o índice inicial da primeira ocorrência da substring na string; gera um erro `ValueError` se não for encontrada. |
| `find` | Retorna a posição do primeiro caractere da **primeira** ocorrência da substring; semelhante a `index`, mas retorna `-1` se não for encontrada. |
| `rfind` | Retorna a posição do primeiro caractere da **última** ocorrência da substring; retorna `-1` se não for encontrada. |
| `replace` | Substitui todas as ocorrências de uma substring por outra string. |
| `strip`, `rstrip`, `lstrip` | Remove espaços em branco (incluindo quebras de linha) dos dois lados (`strip`), do lado direito (`rstrip`) ou do lado esquerdo (`lstrip`). |
| `split` | Divide a string em uma lista de substrings usando um delimitador especificado. |
| `lower` | Converte todos os caracteres alfabéticos para minúsculas. |
| `upper` | Converte todos os caracteres alfabéticos para maiúsculas. |
| `casefold` | Converte caracteres para minúsculas e aplica conversões específicas de região para uniformizar comparações. |
| `ljust`, `rjust` | Alinha o texto à esquerda (`ljust`) ou à direita (`rjust`), preenchendo o lado oposto com espaços (ou outro caractere especificado) para atingir uma largura mínima. |

### 4.2. Expressões Regulares

Expressões regulares oferecem uma maneira flexível de pesquisar ou encontrar padrões de strings (geralmente mais complexos) em texto. Uma única expressão, comumente chamada de `regex`, é uma string formada de acordo com a linguagem de expressões regulares. O módulo re integrado do Python é responsável por aplicar expressões regulares a strings; veremos alguns exemplos de seu uso aqui.

As funções do módulo re se dividem em três categorias: correspondência de padrões, substituição e divisão. Naturalmente, todas elas estão relacionadas; uma `regex` descreve um padrão a ser localizado no texto, que pode ser usado para diversos fins. Vejamos um exemplo simples: suponha que quiséssemos dividir uma string com um número variável de caracteres de espaço em branco (tabulações, espaços e quebras de linha).

A expressão regular que descreve um ou mais caracteres de espaço em branco é `\s+`:

In [13]:
import re

text = 'foo     bar\t baz  \tqux'
print(text, '\n')

re.split(r'\s+', text)

foo     bar	 baz  	qux 



['foo', 'bar', 'baz', 'qux']

Ao chamar `re.split(r"\s+", text)`, a expressão regular é compilada primeiro e, em seguida, seu método `split` é chamado com base no texto passado. Você pode compilar a `regex` com `re.compile`, formando um objeto `regex` reutilizável:

In [14]:
regex = re.compile(r'\s+')
print(regex)

regex.split(text)

re.compile('\\s+')


['foo', 'bar', 'baz', 'qux']

Se, em vez disso, você quiser obter uma lista de todos os padrões correspondentes à `regex`, você pode usar o método `findall`:

In [15]:
regex.findall(text)

['     ', '\t ', '  \t']

Criar um objeto regex com `re.compile` é altamente recomendado se você pretende aplicar a mesma expressão a muitas strings; isso economizará ciclos de CPU.

`match` e `search` estão intimamente relacionados a `findall`. Enquanto `findall` retorna todas as correspondências em uma string, `search` retorna apenas a primeira correspondência. De forma mais rígida, `match` só corresponde ao início da string. Como um exemplo menos trivial, consideremos um bloco de texto e uma expressão regular capaz de identificar a maioria dos endereços de e-mail:

In [16]:
text = """Dave dave@google.com
Steve steve@gmail.com
Rob rob@gmail.com
Ryan ryan@yahoo.com"""

pattern = r'[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,4}'
regex = re.compile(pattern, flags=re.IGNORECASE) # expressão insensível a maiúsculas e minúsculas

Usar `findall` no texto produz uma lista de endereços de e-mail:

In [17]:
regex.findall(text)

['dave@google.com', 'steve@gmail.com', 'rob@gmail.com', 'ryan@yahoo.com']

`search` retorna um objeto de correspondência especial para o primeiro endereço de e-mail no texto. Para a expressão regular anterior, o objeto de correspondência só pode nos informar a posição inicial e final do padrão na string:

In [18]:
re_search = regex.search(text)
re_search

<re.Match object; span=(5, 20), match='dave@google.com'>

In [19]:
text[re_search.start() : re_search.end()]

'dave@google.com'

`regex.match` retorna `None`, pois corresponderá somente se o padrão ocorrer no início da string:

In [21]:
print(regex.match(text))

None


Da mesma forma, sub retornará uma nova string com ocorrências do padrão substituídas por uma nova string:

In [22]:
print(regex.sub('REDACTED', text))

Dave REDACTED
Steve REDACTED
Rob REDACTED
Ryan REDACTED


Suponha que você queira encontrar endereços de e-mail e, simultaneamente, segmentar cada endereço em seus três componentes: nome de usuário, nome de domínio e sufixo de domínio. Para isso, coloque parênteses em torno das partes do padrão a serem segmentadas:

In [23]:
pattern = r'([A-Z0-9._%+-]+)@([A-Z0-9.-]+)\.([A-Z]{2,4})'
regex = re.compile(pattern, flags=re.IGNORECASE)

Um objeto de correspondência produzido por esta regex modificada retorna uma tupla dos componentes do padrão com seu método groups:

In [24]:
re_match = regex.match('wes@bright.net')
re_match.groups()

('wes', 'bright', 'net')

`findall` retorna uma lista de tuplas quando o padrão tem grupos:

In [25]:
regex.findall(text)

[('dave', 'google', 'com'),
 ('steve', 'gmail', 'com'),
 ('rob', 'gmail', 'com'),
 ('ryan', 'yahoo', 'com')]

`sub` também tem acesso aos grupos em cada correspondência usando símbolos especiais como `\1` e `\2`. O símbolo `\1` corresponde ao primeiro grupo correspondente, `\2` corresponde ao segundo e assim por diante:

In [26]:
print(regex.sub(r'Username: \1, Domain: \2, Suffix: \3', text))

Dave Username: dave, Domain: google, Suffix: com
Steve Username: steve, Domain: gmail, Suffix: com
Rob Username: rob, Domain: gmail, Suffix: com
Ryan Username: ryan, Domain: yahoo, Suffix: com


A tabale abaixo apresenta um pequeno sumário das funcões de manipulação de expressões regulares.

| Método | Descrição |
|---------|------------|
| `findall` | Retorna todas as correspondências de um padrão em uma string como uma lista (sem sobreposição). |
| `finditer` | Semelhante a `findall`, mas retorna um iterador em vez de uma lista. |
| `match` | Verifica se o padrão corresponde ao **início** da string e, opcionalmente, separa o padrão em grupos; retorna um objeto `Match` se houver correspondência, caso contrário, `None`. |
| `search` | Procura o padrão em qualquer parte da string (não apenas no início) e retorna um objeto `Match` se encontrado. |
| `split` | Divide a string em partes em cada ocorrência do padrão. |
| `sub`, `subn` | Substitui todas (`sub`) ou as primeiras *n* ocorrências (`subn`) de um padrão na string por uma expressão de substituição; usa os símbolos `\1`, `\2`, ... para se referir a grupos correspondentes no padrão. |

### 4.3. Funções de String em Pandas

Limpar um conjunto de dados confuso para análise geralmente requer muita manipulação de strings. Para complicar ainda mais, uma coluna contendo strings às vezes pode apresentar dados ausentes:

In [27]:
import pandas as pd
import numpy as np

data = {'Dave': 'dave@google.com',
        'Steve': 'steve@gmail.com',
        'Rob': 'rob@gmail.com',
        'Wes': np.nan}

data = pd.Series(data)
data

Dave     dave@google.com
Steve    steve@gmail.com
Rob        rob@gmail.com
Wes                  NaN
dtype: str

In [28]:
data.isna()

Dave     False
Steve    False
Rob      False
Wes       True
dtype: bool

Métodos de string e expressões regulares podem ser aplicados (passando uma função lambda ou outra função) a cada valor usando `data.map`, mas falharão nos valores `NA` (nulos). Para lidar com isso, o `Series` possui métodos orientados a array para operações de string que ignoram e propagam valores `NA`. Eles são acessados ​​por meio do atributo `str` do `Series`; por exemplo, poderíamos verificar se cada endereço de e-mail contém `"gmail"` com `str.contains`:

In [29]:
data.str.contains('gmail')

Dave     False
Steve     True
Rob       True
Wes      False
dtype: bool

O Pandas tem tipos de extensão que fornecem tratamento especializado de strings, inteiros e dados booleanos que até recentemente apresentavam algumas arestas ao trabalhar com dados ausentes:

In [30]:
data_as_string_ext = data.astype('string')
data_as_string_ext

Dave     dave@google.com
Steve    steve@gmail.com
Rob        rob@gmail.com
Wes                 <NA>
dtype: string

In [31]:
data_as_string_ext.str.contains('gmail')

Dave     False
Steve     True
Rob       True
Wes       <NA>
dtype: boolean

Expressões regulares também podem ser usadas, junto com quaisquer opções `re` como `IGNORECASE`:

In [32]:
import re

pattern = r'([A-Z0-9._%+-]+)@([A-Z0-9.-]+)\.([A-Z]{2,4})'
data.str.findall(pattern, flags=re.IGNORECASE)

Dave     [(dave, google, com)]
Steve    [(steve, gmail, com)]
Rob        [(rob, gmail, com)]
Wes                        NaN
dtype: object

Existem algumas maneiras de recuperar elementos vetorizados. Use `str.get` ou indexe no atributo `str`:

In [33]:
matches = data.str.findall(pattern, flags=re.IGNORECASE).str[0]
matches

Dave     (dave, google, com)
Steve    (steve, gmail, com)
Rob        (rob, gmail, com)
Wes                      NaN
dtype: object

In [34]:
matches.str.get(1)

Dave     google
Steve     gmail
Rob       gmail
Wes         NaN
dtype: object

Você pode fatiar strings de forma semelhante usando esta sintaxe:

In [35]:
data.str[:5]

Dave     dave@
Steve    steve
Rob      rob@g
Wes        NaN
dtype: str

O método `str.extract` retornará os grupos capturados de uma expressão regular como um `DataFrame`:

In [36]:
data.str.extract(pattern, flags=re.IGNORECASE)

,0,1,2
Dave,dave,google,com
Steve,steve,gmail,com
Rob,rob,gmail,com
Wes,NaN,NaN,NaN


Veja a tabela abaixo para mais métodos de string do Pandas.

| Método | Descrição |
|---------|------------|
| `cat` | Concatena strings elemento a elemento, com um delimitador opcional. |
| `contains` | Retorna um array booleano indicando se cada string contém o padrão/expressão regular especificado. |
| `count` | Conta o número de ocorrências do padrão. |
| `extract` | Usa uma expressão regular com grupos para extrair uma ou mais strings de uma `Series`; o resultado é um `DataFrame` com uma coluna por grupo. |
| `endswith` | Equivalente a `x.endswith(padrao)` para cada elemento. |
| `startswith` | Equivalente a `x.startswith(padrao)` para cada elemento. |
| `findall` | Retorna uma lista com todas as ocorrências do padrão/regex em cada string. |
| `get` | Retorna o elemento pelo índice especificado (recupera o elemento de posição *i*). |
| `isalnum` | Equivalente a `str.isalnum()` — verifica se todos os caracteres são alfanuméricos. |
| `isalpha` | Equivalente a `str.isalpha()` — verifica se todos os caracteres são letras. |
| `isdecimal` | Equivalente a `str.isdecimal()` — verifica se todos os caracteres são decimais. |
| `isdigit` | Equivalente a `str.isdigit()` — verifica se todos os caracteres são dígitos. |
| `islower` | Equivalente a `str.islower()` — verifica se todos os caracteres estão em minúsculas. |
| `isnumeric` | Equivalente a `str.isnumeric()` — verifica se todos os caracteres são numéricos. |
| `isupper` | Equivalente a `str.isupper()` — verifica se todos os caracteres estão em maiúsculas. |
| `join` | Junta as strings em cada elemento da `Series` usando o separador especificado. |
| `len` | Calcula o comprimento de cada string. |
| `lower`, `upper` | Converte o caso das letras; equivalente a `x.lower()` ou `x.upper()` para cada elemento. |
| `match` | Usa `re.match()` com a expressão regular passada em cada elemento, retornando `True` ou `False` se houver correspondência. |
| `pad` | Adiciona espaços (ou outro caractere) à esquerda, direita ou ambos os lados das strings. |
| `center` | Equivalente a `pad(side="both")` — centraliza o texto. |
| `repeat` | Duplica os valores; por exemplo, `s.str.repeat(3)` é equivalente a `x * 3` para cada string. |
| `replace` | Substitui ocorrências de um padrão/regex por outra string. |
| `slice` | Fatia cada string da `Series` (como em `str[start:end]`). |
| `split` | Divide as strings com base em um delimitador ou expressão regular. |
| `strip` | Remove espaços em branco dos dois lados das strings, incluindo quebras de linha. |
| `rstrip` | Remove espaços em branco à direita. |
| `lstrip` | Remove espaços em branco à esquerda. |

## 5. Dados Categóricos

Esta seção apresenta o tipo `Categorical` do Pandas. Veremos como você pode obter melhor desempenho e uso de memória em algumas operações do Pandas utilizando-o. Também apresentarei algumas ferramentas que podem ajudar no uso de dados categóricos em aplicações estatísticas e de aprendizado de máquina.

### 5.1. Histórico e Motivação

Frequentemente, uma coluna em uma tabela pode conter instâncias repetidas de um conjunto menor de valores distintos. Já vimos funções como `unique` e `value_counts`, que nos permitem extrair os valores distintos de um array e calcular suas frequências, respectivamente:

In [37]:
values = pd.Series(['apple', 'orange', 'apple', 'apple'] * 2)
values

0     apple
1    orange
2     apple
3     apple
4     apple
5    orange
6     apple
7     apple
dtype: str

In [39]:
pd.unique(values)

<ArrowStringArray>
['apple', 'orange']
Length: 2, dtype: str

In [40]:
pd.Series(values).value_counts()

apple     6
orange    2
Name: count, dtype: int64

Muitos sistemas de dados (para data warehouse, estatística computacional ou outros usos) desenvolveram abordagens especializadas para representar dados com valores repetidos, visando armazenamento e computação mais eficientes. Em data warehouse, uma prática recomendada é usar as chamadas tabelas de dimensões, contendo os valores distintos e armazenando as observações primárias como chaves inteiras que referenciam a tabela de dimensões:

In [41]:
values = pd.Series([0, 1, 0, 0] * 2)
dim = pd.Series(['apple', 'orange'])

values

0    0
1    1
2    0
3    0
4    0
5    1
6    0
7    0
dtype: int64

In [42]:
dim

0     apple
1    orange
dtype: str

Podemos usar o método `take` para restaurar a série original de strings:

In [43]:
dim.take(values)

0     apple
1    orange
0     apple
0     apple
0     apple
1    orange
0     apple
0     apple
dtype: str

Essa representação em números inteiros é chamada de representação categórica ou codificada por dicionário. A matriz de valores distintos pode ser chamada de categorias, dicionário ou níveis dos dados. Neste curso, usaremos os termos categórico e categorias. Os valores inteiros que fazem referência às categorias são chamados de códigos de categoria ou simplesmente códigos.

A representação categórica pode gerar melhorias significativas de desempenho ao realizar análises. Você também pode realizar transformações nas categorias, deixando os códigos inalterados. Alguns exemplos de transformações que podem ser feitas a um custo relativamente baixo são:

- Renomear categorias

- Adicionar uma nova categoria sem alterar a ordem ou a posição das categorias existentes

### 5.2. Tipo de Extensão Categórica em Pandas

O Pandas possui um tipo especial de extensão `Categorical` para armazenar dados que utilizam a representação ou codificação categórica baseada em números inteiros. Esta é uma técnica popular de compressão de dados para dados com muitas ocorrências de valores semelhantes e pode fornecer desempenho significativamente mais rápido com menor uso de memória, especialmente para dados de string.

Vamos considerar o exemplo da `Series` anterior:

In [44]:
fruits = ['apple', 'orange', 'apple', 'apple'] * 2
fruits

['apple', 'orange', 'apple', 'apple', 'apple', 'orange', 'apple', 'apple']

In [46]:
N = len(fruits)

rng = np.random.default_rng(seed=42)
frame = pd.DataFrame({'fruit': fruits,
                      'basket_id': np.arange(N),
                      'count': rng.integers(3, 15, size=N),
                      'weight': rng.uniform(0, 4, size=N)},
                      columns=['basket_id', 'fruit', 'count', 'weight'])

frame

,basket_id,fruit,count,weight
0,0,apple,4,0.376709
1,1,orange,12,3.902489
2,2,apple,10,3.044559
3,3,apple,8,3.144257
4,4,apple,8,0.512455
5,5,orange,13,1.801544
6,6,apple,4,1.483192
7,7,apple,11,3.707060


Aqui, `frame['fruit']` é um array de objetos string do Python. Podemos convertê-lo para categórico chamando:

In [47]:
fruit_cat = frame['fruit'].astype('category')
fruit_cat

0     apple
1    orange
2     apple
3     apple
4     apple
5    orange
6     apple
7     apple
Name: fruit, dtype: category
Categories (2, str): ['apple', 'orange']

Os valores para `fruit_cat` agora são uma instância de `pandas.Categorical`, que você pode acessar por meio do atributo `.array`:

In [48]:
c = fruit_cat.array
c

['apple', 'orange', 'apple', 'apple', 'apple', 'orange', 'apple', 'apple']
Categories (2, str): ['apple', 'orange']

In [49]:
type(c)

pandas.Categorical

O objeto `Categorical` possui atributos de categorias e códigos:

In [50]:
c.categories

Index(['apple', 'orange'], dtype='str')

In [51]:
c.codes

array([0, 1, 0, 0, 0, 1, 0, 0], dtype=int8)

Eles podem ser acessados ​​mais facilmente usando o acessador `cat`, que será explicado em breve na subseção `"Métodos Categóricos"`.

Um truque útil para obter um mapeamento entre códigos e categorias é:

In [53]:
dict(enumerate(c.categories))

{0: 'apple', 1: 'orange'}

Você pode converter uma coluna `DataFrame` em categórica atribuindo o resultado convertido:

In [54]:
frame['fruit'] = frame['fruit'].astype('category')
frame['fruit']

0     apple
1    orange
2     apple
3     apple
4     apple
5    orange
6     apple
7     apple
Name: fruit, dtype: category
Categories (2, str): ['apple', 'orange']

Você também pode criar `pandas.Categorical` diretamente de outros tipos de sequências Python:

In [55]:
my_categories = pd.Categorical(['foo', 'bar', 'baz', 'foo', 'bar'])
my_categories

['foo', 'bar', 'baz', 'foo', 'bar']
Categories (3, str): ['bar', 'baz', 'foo']

Se você obteve dados codificados categóricos de outra fonte, pode usar o construtor alternativo `from_codes`:

In [56]:
categories = ['foo', 'bar', 'baz']
codes = [0, 1, 2, 0, 0, 1]

my_cats = pd.Categorical.from_codes(codes, categories)
my_cats

['foo', 'bar', 'baz', 'foo', 'foo', 'bar']
Categories (3, str): ['foo', 'bar', 'baz']

A menos que explicitamente especificado, as conversões categóricas não pressupõem uma ordenação específica das categorias. Portanto, o array `categories` pode estar em uma ordem diferente dependendo da ordenação dos dados de entrada. Ao usar `from_codes` ou qualquer um dos outros construtores, você pode indicar que as categorias têm uma ordenação significativa:

In [57]:
ordered_cat = pd.Categorical.from_codes(codes, categories, ordered=True)
ordered_cat

['foo', 'bar', 'baz', 'foo', 'foo', 'bar']
Categories (3, str): ['foo' < 'bar' < 'baz']

A saída `[foo < bar < baz]` indica que `'foo'` precede `'bar'` na ordenação, e assim por diante. Uma instância categórica não ordenada pode ser ordenada com `as_ordered`:

In [58]:
my_cats.as_ordered()

['foo', 'bar', 'baz', 'foo', 'foo', 'bar']
Categories (3, str): ['foo' < 'bar' < 'baz']

Por fim, dados categóricos não precisam ser strings, embora tenhamos mostrado apenas exemplos de strings. Uma matriz categórica pode consistir em qualquer tipo de valor imutável.

### 5.3. Cálculos com Categóricos

O uso de `Categorical` no `Pandas`, em comparação com a versão não codificada (como um array de strings), geralmente se comporta da mesma maneira. Algumas partes do Pandas, como a função `groupby`, têm melhor desempenho ao trabalhar com categóricos. Existem também algumas funções que podem utilizar o sinalizador `ordered`.

Vamos considerar alguns dados numéricos aleatórios e usar a função de agrupamento (binning) `pandas.qcut`. Isso retorna `pandas.Categorical`; usamos `pandas.cut` anteriormente, mas ignoramos os detalhes de como os categóricos funcionam:

In [ ]:
rng = np.random.default_rng(seed=42)

draws = rng.standard_normal(1000)
draws[:5]

array([ 0.30471708, -1.03998411,  0.7504512 ,  0.94056472, -1.95103519])

Vamos calcular um agrupamento de quartis desses dados e extrair algumas estatísticas:

In [2]:
bins = pd.qcut(draws, 4)
bins

[(0.00618, 0.59], (-3.649, -0.696], (0.59, 3.179], (0.59, 3.179], (-3.649, -0.696], ..., (0.00618, 0.59], (0.59, 3.179], (0.00618, 0.59], (0.00618, 0.59], (0.59, 3.179]]
Length: 1000
Categories (4, interval[float64, right]): [(-3.649, -0.696] < (-0.696, 0.00618] < (0.00618, 0.59] < (0.59, 3.179]]

Embora úteis, os quartis exatos da amostra podem ser menos úteis para a produção de um relatório do que os nomes dos quartis. Podemos fazer isso com o argumento labels para `qcut`:

In [3]:
bins = pd.qcut(draws, 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
bins

['Q3', 'Q1', 'Q4', 'Q4', 'Q1', ..., 'Q3', 'Q4', 'Q3', 'Q3', 'Q4']
Length: 1000
Categories (4, str): ['Q1' < 'Q2' < 'Q3' < 'Q4']

Os `bins` rotulados como categóricos não contêm informações sobre as bordas dos `bins` nos dados, então podemos usar `groupby` para extrair algumas estatísticas descritivas:

In [7]:
bins = pd.Series(bins, name='quartile')

results = (pd.Series(draws).groupby(bins).agg(['count', 'min', 'max']).reset_index())
results

,quartile,count,min,max
0,Q1,250,-3.648413,-0.697424
1,Q2,250,-0.695943,0.006017
2,Q3,250,0.006339,0.589547
3,Q4,250,0.590906,3.178854


A coluna `'quartil'` no resultado retém as informações categóricas originais, incluindo a ordenação, dos grupos:

In [8]:
results['quartile']

0    Q1
1    Q2
2    Q3
3    Q4
Name: quartile, dtype: category
Categories (4, str): ['Q1' < 'Q2' < 'Q3' < 'Q4']

Como mencionado no início da seção, tipos categóricos podem melhorar o desempenho e o uso de memória, então vamos ver alguns exemplos. Considere algumas séries com 10 milhões de elementos e um pequeno número de categorias distintas:

In [9]:
N = 10_000_000

labels = pd.Series(['foo', 'bar', 'baz', 'qux'] * (N // 4))

Agora convertemos rótulos em categóricos:

In [10]:
categories = labels.astype('category')

Agora notamos que os rótulos usam significativamente mais memória do que as categorias:

In [11]:
labels.memory_usage(deep=True)

110000132

In [12]:
categories.memory_usage(deep=True)

10000177

A conversão para categoria não é gratuita, é claro, mas é um custo único:

In [13]:
%time _ = labels.astype('category')

CPU times: user 114 ms, sys: 55.9 ms, total: 170 ms
Wall time: 170 ms


As operações `"GroupBy"` podem ser significativamente mais rápidas com categóricas porque os algoritmos subjacentes usam a matriz de códigos baseada em números inteiros em vez de uma matriz de strings. Aqui, comparamos o desempenho de `value_counts()`, que usa internamente a mecânica `"GroupBy"`:

In [14]:
%timeit labels.value_counts()

80 ms ± 2.19 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [15]:
%timeit categories.value_counts()

29 ms ± 2.6 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


### 5.4. Métodos Categóricos

`Series` contendo dados categóricos possuem vários métodos especiais semelhantes aos métodos de string especializados `Series.str`. Isso também fornece acesso conveniente às categorias e códigos. Considere a `Series`:

In [16]:
s = pd.Series(['a', 'b', 'c', 'd'] * 2)

cat_s = s.astype('category')
cat_s

0    a
1    b
2    c
3    d
4    a
5    b
6    c
7    d
dtype: category
Categories (4, str): ['a', 'b', 'c', 'd']

O atributo de acesso especial `cat` fornece acesso a métodos categóricos:

In [17]:
cat_s.cat.codes

0    0
1    1
2    2
3    3
4    0
5    1
6    2
7    3
dtype: int8

In [18]:
cat_s.cat.categories

Index(['a', 'b', 'c', 'd'], dtype='str')

Suponha que sabemos que o conjunto real de categorias para esses dados se estende além dos quatro valores observados nos dados. Podemos usar o método `set_categories` para alterá-los:

In [19]:
actual_categories = ['a', 'b', 'c', 'd', 'e']

cat_s2 = cat_s.cat.set_categories(actual_categories)
cat_s2

0    a
1    b
2    c
3    d
4    a
5    b
6    c
7    d
dtype: category
Categories (5, str): ['a', 'b', 'c', 'd', 'e']

Embora pareça que os dados não foram alterados, as novas categorias serão refletidas nas operações que as utilizam. Por exemplo, `value_counts` respeita as categorias, se presentes:

In [20]:
cat_s.value_counts()

a    2
b    2
c    2
d    2
Name: count, dtype: int64

In [21]:
cat_s2.value_counts()

a    2
b    2
c    2
d    2
e    0
Name: count, dtype: int64

Em grandes conjuntos de dados, categorias são frequentemente usadas como uma ferramenta conveniente para economizar memória e melhorar o desempenho. Após filtrar um `DataFrame` ou `Series` grande, muitas das categorias podem não aparecer nos dados. Para ajudar com isso, podemos usar o método `remove_unused_categories` para cortar categorias não observadas:

In [22]:
cat_s3 = cat_s[cat_s.isin(['a', 'b'])]
cat_s3

0    a
1    b
4    a
5    b
dtype: category
Categories (4, str): ['a', 'b', 'c', 'd']

In [23]:
cat_s3.cat.remove_unused_categories()

0    a
1    b
4    a
5    b
dtype: category
Categories (2, str): ['a', 'b']

A tabela abaixo apresenta uma lista de métodos categóricos disponíveis.

| Método | Descrição |
|---------|------------|
| `add_categories` | Adiciona novas categorias (não utilizadas) ao final das categorias existentes. |
| `as_ordered` | Define as categorias como ordenadas. |
| `as_unordered` | Define as categorias como não ordenadas. |
| `remove_categories` | Remove categorias, definindo como nulos os valores que pertenciam a essas categorias. |
| `remove_unused_categories` | Remove categorias que não aparecem nos dados. |
| `rename_categories` | Substitui as categorias pelo conjunto indicado de novos nomes; não é possível alterar o número de categorias. |
| `reorder_categories` | Funciona como `rename_categories`, mas também pode alterar o resultado para ter categorias ordenadas. |
| `set_categories` | Substitui as categorias pelo conjunto indicado de novas categorias; pode adicionar ou remover categorias. |

Ao usar ferramentas de estatística ou aprendizado de máquina, você frequentemente transformará dados categóricos em variáveis ​​fictícias (*dummy variables*), também conhecidas como codificação *one-hot*. Isso envolve a criação de um `DataFrame` com uma coluna para cada categoria distinta; essas colunas contêm `1`s para ocorrências de uma determinada categoria e `0` para as demais.

Considere o exemplo anterior:

In [24]:
cat_s = pd.Series(['a', 'b', 'c', 'd'] * 2, dtype='category')
cat_s

0    a
1    b
2    c
3    d
4    a
5    b
6    c
7    d
dtype: category
Categories (4, str): ['a', 'b', 'c', 'd']

Conforme mencionado anteriormente nesta seção, a função `pandas.get_dummies` converte esses dados categóricos unidimensionais em um `DataFrame` contendo a variável fictícia:

In [25]:
pd.get_dummies(cat_s, dtype=int)

,a,b,c,d
0,1,0,0,0
1,0,1,0,0
2,0,0,1,0
3,0,0,0,1
4,1,0,0,0
5,0,1,0,0
6,0,0,1,0
7,0,0,0,1


In [26]:
pd.get_dummies(cat_s) # sem informar o dtype o valor padrão é bool

,a,b,c,d
0,True,False,False,False
1,False,True,False,False
2,False,False,True,False
3,False,False,False,True
4,True,False,False,False
5,False,True,False,False
6,False,False,True,False
7,False,False,False,True


**Nota**! A preparação eficaz de dados pode melhorar significativamente a produtividade, permitindo que você gaste mais tempo analisando dados e menos tempo preparando-os para análise.

## 6. Exercícios

In [27]:
# Esta célula informa ao Jupyter para fornecer informações detalhadas de depuração
# quando ocorrer um erro de execução. Execute-o antes de trabalhar nos exercícios.

%xmode Verbose

Exception reporting mode: Verbose


In [28]:
import pandas as pd
import numpy as np

### 6.1. Exercício

Dada uma `Series` com valores `[1.2, -3.5, np.nan, 0]`, identifique quais elementos são `NA` e conte-os.
- Dica: use `isna()` e `sum()`.

In [30]:
ex1 = pd.Series([1.2, -3.5, np.nan, 0])

print(ex1.isna().sum())

1


### 6.2. Exercício

Para o `DataFrame` abaixo, remova todas as linhas que contenham pelo menos um `NA`.
- Dica: `dropna()`.

In [31]:
data = pd.DataFrame([[1., 6.5, 3.],
                     [1., np.nan, np.nan],
                     [np.nan, np.nan, np.nan],
                     [np.nan, 6.5, 3.]])

data.dropna()

,0,1,2
0,1.0,6.5,3.0


### 6.3. Exercício

Usando o mesmo `DataFrame` do exercício anterior, remova apenas as linhas que sejam totalmente `NA`.
- Dica: `dropna(how="all")`.

In [32]:
data.dropna(how='all')

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
3,NaN,6.5,3.0


### 6.4. Exercício

Crie um `DataFrame` com `7` linhas e `4` colunas com valores numéricos e `NA`, mantenha apenas as linhas que têm pelo menos `2` valores não-`NA`.
- Dica: `dropna(thresh=3)`.


In [33]:
data = pd.DataFrame({"A": [1.0, np.nan, 3.0, np.nan, 5.0, 6.0, np.nan],
                     "B": [np.nan, 2.0, 3.0, 4.0, np.nan, 6.0, 7.0],
                     "C": [1.0, 2.0, np.nan, 4.0, 5.0, np.nan, 7.0],
                     "D": [np.nan, np.nan, 3.0, 4.0, 5.0, 6.0, 7.0]})

data.dropna(thresh=3)

,A,B,C,D
2,3.0,3.0,NaN,3.0
3,NaN,4.0,4.0,4.0
4,5.0,NaN,5.0,5.0
5,6.0,6.0,NaN,6.0
6,NaN,7.0,7.0,7.0


### 6.5. Exercício

Criei uma série numérica e com valores `NA`. Em seguida, impute (substitua) os valores `NA` com a média da série.
- Dica: `fillna(series.mean())`.

In [34]:
series = pd.Series([1.0, 4.5, 3.4, 3.7, np.nan, 7.8, np.nan, 6.5, 5.6, np.nan])

series = series.fillna(series.mean())
print(f"Média: {series.mean()}")
series

Média: 4.642857142857143


0    1.000000
1    4.500000
2    3.400000
3    3.700000
4    4.642857
5    7.800000
6    4.642857
7    6.500000
8    5.600000
9    4.642857
dtype: float64

### 6.6. Exercício

Utilizando o `DataFrame` do exercício `6.4`. Impute (substitua) `NA`s usando `forward-fill` até 2 posições consecutivas.
- Dica: `ffill(limit=2)`.

In [35]:
data = pd.DataFrame({"A": [1.0, np.nan, 3.0, np.nan, 5.0, 6.0, np.nan],
                     "B": [np.nan, 2.0, 3.0, 4.0, np.nan, 6.0, 7.0],
                     "C": [1.0, 2.0, np.nan, 4.0, 5.0, np.nan, 7.0],
                     "D": [np.nan, np.nan, 3.0, 4.0, 5.0, 6.0, 7.0]})

data.ffill(limit=2)

,A,B,C,D
0,1.0,NaN,1.0,NaN
1,1.0,2.0,2.0,NaN
2,3.0,3.0,2.0,3.0
3,3.0,4.0,4.0,4.0
4,5.0,4.0,5.0,5.0
5,6.0,6.0,5.0,6.0
6,6.0,7.0,7.0,7.0


### 6.7. Exercício

Dado um `DataFrame` com linhas duplicadas, gere uma versão sem duplicatas considerando apenas a coluna `"k1"` e mantendo a última ocorrência.
- Dica: `drop_duplicates(subset=["k1"], keep="last")`.

In [37]:
data = pd.DataFrame({'k1': ['one', 'two'] * 2 + ['two'] * 3,
                     'k2': [1, 1, 2, 3, 3, 4, 4]})

data = data.drop_duplicates(subset=['k1'], keep='last')
data

,k1,k2
2,one,2
6,two,4


### 6.8. Exercício

Construa uma coluna nova `"animal"` a partir da coluna `"food"` usando um dicionário de mapeamento.
- Dica: `Series.map(mapping)`.

In [38]:
data = pd.DataFrame({'food': ['bacon', 'pulled pork', 'bacon',
                              'pastrami', 'corned beef', 'bacon',
                              'pastrami', 'honey ham', 'nova lox'],
                    'ounces': [4, 3, 12, 6, 7.5, 8, 3, 5, 6]
                    })

mapping = {'bacon': 'pig', 'pulled pork': 'pig', 'pastrami': 'cow', 'corned beef': 'cow', 'honey ham': 'pig', 'nova lox': 'salmon'}
data['animal'] = data['food'].map(mapping)
data

,food,ounces,animal
0,bacon,4.0,pig
1,pulled pork,3.0,pig
2,bacon,12.0,pig
3,pastrami,6.0,cow
4,corned beef,7.5,cow
5,bacon,8.0,pig
6,pastrami,3.0,cow
7,honey ham,5.0,pig
8,nova lox,6.0,salmon


### 6.9. Exercício

Substitua valores sentinela `[-999, -1000]` em uma `Series` por `np.nan`, usando um único comando.
- Dica: `replace(...)`.

In [40]:
data = pd.Series([1., -999., 2., -999., -1000., 3.])

data = data.replace([-999, -1000], np.nan)
data

0    1.0
1    NaN
2    2.0
3    NaN
4    NaN
5    3.0
dtype: float64

### 6.10. Exercício

Renomeie os índices de um `DataFrame` aplicando uma função que corta os primeiros `4` caracteres e converte para maiúsculas.
- Dica: `index.map(func)` ou `rename(index=...)`.

In [43]:
def up_four(x):
    return x[:4].upper().strip()

data = pd.DataFrame(np.arange(12).reshape((3, 4)),
                    index=['Ohio', 'Colorado', 'New York'],
                    columns=['one', 'two', 'three', 'four'])

data.index = data.index.map(up_four)
data

,one,two,three,four
OHIO,0,1,2,3
COLO,4,5,6,7
NEW,8,9,10,11


### 6.11. Exercício

Agrupe a lista de idades `[20,22,25,27,21,23,37,31,61,45,41,32]` em bins `[18,25,35,60,100]` e conte quantos itens há em cada bin.
- Dica: `pd.cut(...)` e `value_counts()`.


In [44]:
ages = [20,22,25,27,21,23,37,31,61,45,41,32]
bins = [18, 25, 35, 60, 100]

age_cats = pd.cut(ages, bins)
pd.Series(age_cats).value_counts()

(18, 25]     5
(25, 35]     3
(35, 60]     3
(60, 100]    1
Name: count, dtype: int64

### 6.12. Exercício

Em um `DataFrame` com distribuições aproximadamente normais, encontre todas as linhas que contenham algum valor com `|valor| > 3` e substitua esses valores por `sinal(valor)*3`.
- Dica: `(data.abs()>3).any(axis="columns")` e `np.sign`.

In [49]:
data = pd.DataFrame(np.random.standard_normal((1000, 4)))

data[(data.abs() > 3).any(axis='columns')].head()

,0,1,2,3
3,-1.007057,3.014037,-1.188546,-2.010805
86,0.423944,0.796665,3.340488,-0.195280
87,-1.133831,-0.823036,3.322787,0.448278
166,0.174508,-1.555813,-3.013580,0.289818
250,-0.331513,-3.386030,1.316140,-1.217355


In [50]:
data[data.abs() > 3] = np.sign(data) * 3

# Máscara para display dos dados modificados
mask = (data.abs() == 3.).any(axis=1)
data[mask]

,0,1,2,3
3,-1.007057,3.000000,-1.188546,-2.010805
86,0.423944,0.796665,3.000000,-0.195280
87,-1.133831,-0.823036,3.000000,0.448278
166,0.174508,-1.555813,-3.000000,0.289818
250,-0.331513,-3.000000,1.316140,-1.217355
258,0.253765,-3.000000,-0.647534,0.334792
298,-1.402352,-0.736521,-3.000000,0.350653
307,-0.994015,-3.000000,-1.014420,1.688537
358,-3.000000,-2.235933,0.782291,-0.274773
415,0.417253,0.680626,-3.000000,0.050901


### 6.13. Exercício

Gere uma amostra aleatória sem substituição de `3` linhas de um `DataFrame` e outra amostra com substituição de tamanho `10` a partir de uma `Series`.
- Dica: `sample(n=3)` e `sample(n=10, replace=True)`.


In [52]:
df = pd.DataFrame(np.arange(5 * 7).reshape(5, 7))

df.sample(n=3)

,0,1,2,3,4,5,6
1,7,8,9,10,11,12,13
2,14,15,16,17,18,19,20
0,0,1,2,3,4,5,6


In [53]:
samp = pd.Series([8, -4, 3, 7, -9])

samp.sample(n=10, replace=True)

4   -9
3    7
0    8
4   -9
0    8
2    3
4   -9
0    8
3    7
1   -4
dtype: int64

### 6.14. Exercício

Dado um `Series` de frutas repetidas, converta para `dtype` `"category"`, adicione uma categoria extra não observada, calcule `value_counts()` incluindo categorias com contagem zero e depois remova categorias não usadas.
- Dica: `astype("category")`, `cat.set_categories([...])`, `value_counts()`, `cat.remove_unused_categories()`.

In [54]:
fruits = pd.Series(['apple', 'orange', 'banana', 'apple', 'banana', 'banana', 'banana', 'apple', 'orange', 'apple'])

fruits_cat = fruits.astype("category")
new_cats = ['apple', 'orange', 'banana', 'melon', 'grapes']
fruits_cat = fruits_cat.cat.set_categories(new_cats)
print(fruits_cat.value_counts())

fruits_cat.cat.remove_unused_categories()

apple     4
banana    4
orange    2
melon     0
grapes    0
Name: count, dtype: int64


0     apple
1    orange
2    banana
3     apple
4    banana
5    banana
6    banana
7     apple
8    orange
9     apple
dtype: category
Categories (3, str): ['apple', 'orange', 'banana']